In [1]:
# import os 
# os.environ["ECCODES_DEFINITION_PATH"] ="grib_tables/definitions.edzw-2.32.0-1/"
import utils
import xarray as xr
import numpy as np

In [6]:
np.concatenate(
        [
            np.arange(0.0, 800.0, 50.0),
            np.arange(800.0, 3000.0, 100.0),
            np.arange(3000.0, 4000.0, 250.0),
            np.arange(4000.0, 6500, 500.0),
        ]
    )

array([   0.,   50.,  100.,  150.,  200.,  250.,  300.,  350.,  400.,
        450.,  500.,  550.,  600.,  650.,  700.,  750.,  800.,  900.,
       1000., 1100., 1200., 1300., 1400., 1500., 1600., 1700., 1800.,
       1900., 2000., 2100., 2200., 2300., 2400., 2500., 2600., 2700.,
       2800., 2900., 3000., 3250., 3500., 3750., 4000., 4500., 5000.,
       5500., 6000.])

In [3]:
dset = utils.get_files_sfc("HZEROCL"

)

2025-08-06 10:29:55 INFO - download_file: Fetching file https://meteohub.mistralportal.it/nwp/ICON-2I_SURFACE_PRESSURE_LEVELS/2025080600/HZEROCL/icon_2I_2025080600_isothermZero-0.grib
2025-08-06 10:30:11 INFO - get_files_sfc: Loading files into xarray


In [4]:
dset

<xarray.Dataset> Size: 169MB
Dimensions:       (step: 73, latitude: 761, longitude: 761)
Coordinates:
    time          datetime64[ns] 8B 2025-08-06
  * step          (step) timedelta64[ns] 584B 00:00:00 ... 3 days 00:00:00
    isothermZero  float64 8B 0.0
  * latitude      (latitude) float64 6kB 33.7 33.72 33.74 ... 48.86 48.88 48.9
  * longitude     (longitude) float64 6kB 3.0 3.025 3.05 ... 21.95 21.97 22.0
    valid_time    (step) datetime64[ns] 584B 2025-08-06 ... 2025-08-09
Data variables:
    HZEROCL       (step, latitude, longitude) float32 169MB 5.059e+03 ... 4.1...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             cnmc
    GRIB_centreDescription:  Rome
    GRIB_subCentre:          255
    Conventions:             CF-1.7
    institution:             Rome
    history:                 2025-08-06T10:30 GRIB to CDM+CF via cfgrib-0.9.1...

In [2]:
dset = utils.get_files_levels(
        vars=["WSHEAR_U", "WSHEAR_V"]
    ).squeeze()

2025-07-18 08:55:39 INFO - download_file: Fetching file https://meteohub.mistralportal.it/nwp/ICON-2I_SURFACE_PRESSURE_LEVELS/2025071800/WSHEAR_U/icon_2I_2025071800_heightAboveGroundLayer-6000.grib
2025-07-18 08:55:39 INFO - download_file: Fetching file https://meteohub.mistralportal.it/nwp/ICON-2I_SURFACE_PRESSURE_LEVELS/2025071800/WSHEAR_V/icon_2I_2025071800_heightAboveGroundLayer-6000.grib
2025-07-18 08:55:39 INFO - get_files_levels: Loading files into xarray


In [8]:
u_shear_cf_name = utils.find_variable_by_long_name(
    dset, "U-component of (vertical) wind shear vector between two levels"
)
v_shear_cf_name = utils.find_variable_by_long_name(
    dset, "V-component of (vertical) wind shear vector between two levels"
)
dset["shear"] = np.sqrt(
    (
        dset[u_shear_cf_name].metpy.convert_units("knots") ** 2
        + dset[v_shear_cf_name].metpy.convert_units("knots") ** 2
    )
).metpy.dequantify()

In [9]:
dset

<xarray.Dataset> Size: 507MB
Dimensions:                 (step: 73, latitude: 761, longitude: 761)
Coordinates:
    time                    datetime64[ns] 8B 2025-07-18
  * step                    (step) timedelta64[ns] 584B 00:00:00 ... 3 days 0...
    heightAboveGroundLayer  float64 8B 6e+03
  * latitude                (latitude) float64 6kB 33.7 33.72 ... 48.88 48.9
  * longitude               (longitude) float64 6kB 3.0 3.025 ... 21.97 22.0
    valid_time              (step) datetime64[ns] 584B 2025-07-18 ... 2025-07-21
Data variables:
    WSHEAR_U                (step, latitude, longitude) float32 169MB 3.219 ....
    WSHEAR_V                (step, latitude, longitude) float32 169MB -3.022 ...
    shear                   (step, latitude, longitude) float32 169MB 8.584 ....
Attributes:
    GRIB_edition:            2
    GRIB_centre:             cnmc
    GRIB_centreDescription:  Rome
    GRIB_subCentre:          255
    Conventions:             CF-1.7
    institution:             Rome
    history:                 2025-07-18T08:55 GRIB to CDM+CF via cfgrib-0.9.1...